# Test Static Workflow (Two-Step: Text → Facts → Dataset)

This notebook tests the static workflow that mirrors a simple two-agent pipeline:
1. **Fact Extractor**: Raw document → structured facts (subject, predicate, object, source_span)
2. **Dataset Builder / Validator**: Facts → structured dataset (DataFrame or schema-conforming records)

In [1]:
import sys
sys.path.insert(0, '..')

from src.experimentutils import get_all_markdown_paths, save_extraction_results_with_timestamp
from src.static_workflow import (
    run_two_step_text_to_dataset,
    extract_facts_from_text,
    build_fact_dataset,
    llm_build_dataset_from_facts,
    build_dataset_from_schema_output,
)
from src.direct_llm_call import read_markdown_file
from src.standards import METADATA_STANDARDS

In [2]:
# Get all paper paths and select first paper
all_paths = get_all_markdown_paths()
print(f"Found {len(all_paths)} papers")

sample_path = all_paths[0]
print(f"Testing with: {sample_path.split('/')[-1]}")

# Read document text
raw_text = read_markdown_file(sample_path)
print(f"Document length: {len(raw_text)} chars")

Found 100 papers
Testing with: 1. Mason 1986 Cassava-cowpea and cassava-peanut intercropping II Leaf afrea index and dry matter accumulation.md
Document length: 21864 chars


In [4]:
# Run two-step workflow (deterministic dataset builder: facts → DataFrame)
out = run_two_step_text_to_dataset(
    text=raw_text,
    max_facts=20,
    use_llm_dataset_builder=False,
)

print(f"Extracted {len(out['facts_result'].facts)} facts")
print(f"Dataset rows: {len(out['dataset'])}")
print(f"Validation OK: {out['validation_ok']}")
if out['validation_issues']:
    print(f"Validation issues: {out['validation_issues']}")

Extracted 20 facts
Dataset rows: 20
Validation OK: True


In [5]:
# Display extracted facts
import pandas as pd
out['dataset'][['subject', 'predicate', 'object']].head(10)

,subject,predicate,object
0,Cassava-cowpea and Cassava-peanut intercropping,influence on leaf area index,not well known
1,Cassava-cowpea and Cassava-peanut intercropping,influence on dry matter production,not well known
2,Cassava-cowpea and Cassava-peanut intercropping,influence on dry matter partitioning,not well known
3,Intercropping systems,often result in,better land use efficiencies than sole croppin...
4,Cassava-cowpea and Cassava-peanut intercroppin...,produced,greater land use efficiencies than sole croppi...
5,Intercropping systems,produced LAI's,0.6 to 1.9 greater than cassava sole cropping ...
6,Intercropping systems,produced LAI's,0.8 to 1.0 lower than sole cropped cassava at ...
7,Cassava-cowpea and Cassava-peanut intercroppin...,produced,42 to 250 g m^-2 more dry matter than sole cro...
8,Dry matter production at end of growing season,was similar for,intercropping systems and sole cropped cassava
9,Intercropping with peanut in 1982,associated with,reduction in cassava LAI of 0.8 and production...


In [6]:
# Optional: Run with LLM dataset builder (facts → schema-conforming output)
# This uses the same schema as direct_llm_call for fair comparison
out_llm = run_two_step_text_to_dataset(
    text=raw_text,
    max_facts=20,
    use_llm_dataset_builder=True,
    dataset_standard=METADATA_STANDARDS["climate_vs_cropyield"],
    dataset_records_key="yield_records",
)

print(f"Extracted {len(out_llm['facts_result'].facts)} facts")
print(f"Schema-built dataset rows: {len(out_llm['dataset'])}")
print(f"Validation OK: {out_llm['validation_ok']}")

Extracted 20 facts
Schema-built dataset rows: 9
Validation OK: False


In [7]:
out_llm['facts_result'].facts

[Fact(subject='Cassava-cowpea and Cassava-peanut intercropping', predicate='influence on leaf area index', object='not well known', source_span='Little is known about the influence of intercropping on leaf area index, dry matter production, or partitioning of dry matter to the harvestable plant part.'),
 Fact(subject='Cassava-cowpea and Cassava-peanut intercropping', predicate='influence on dry matter production', object='not well known', source_span='Little is known about the influence of intercropping on leaf area index, dry matter production, or partitioning of dry matter to the harvestable plant part.'),
 Fact(subject='Cassava-cowpea and Cassava-peanut intercropping', predicate='influence on dry matter partitioning', object='not well known', source_span='Little is known about the influence of intercropping on leaf area index, dry matter production, or partitioning of dry matter to the harvestable plant part.'),
 Fact(subject='Intercropping systems', predicate='often result in', obj

In [8]:
# Display schema-built dataset (from LLM dataset builder step above)
if 'out_llm' in globals() and 'schema_output' in out_llm:
    df_llm = out_llm['dataset']
    cols = [c for c in ['crop_type', 'yield_value', 'location', 'year'] if c in df_llm.columns]
    df_llm[cols].head(10) if cols else df_llm.head(10)
else:
    print("Run the LLM dataset builder cell above first.")

In [9]:
# Save results to CSV (timestamped filename)
# Save facts-as-rows (deterministic output)
saved_path = save_extraction_results_with_timestamp(
    results=out['facts_result'],
    base_name="static_workflow_facts",
    records_key="facts",
    include_time=False,
)
print(f"Saved facts to: {saved_path}")

# Save schema-built dataset (only if you ran the LLM dataset builder cell above)
if 'out_llm' in globals() and 'schema_output' in out_llm:
    saved_path_llm = save_extraction_results_with_timestamp(
        results=out_llm['schema_output'],
        base_name="static_workflow_schema_dataset",
        records_key="yield_records",
        include_time=False,
    )
    print(f"Saved schema-built dataset to: {saved_path_llm}")

Saved facts to: /home/com3dian/Github/meta_analysis_agents/outputs/2026-03-10/static_workflow_facts_2026-03-10.csv
Saved schema-built dataset to: /home/com3dian/Github/meta_analysis_agents/outputs/2026-03-10/static_workflow_schema_dataset_2026-03-10.csv
